### Step 4: Formatting and Vectorizing for Training

This section loads the synthetic dataset, formats it, and vectorizes the queries using `fastembed`.

In [9]:
import pandas as pd
import json
from fastembed import TextEmbedding
import numpy as np

# 1. Load your collected dataset
with open('../data/router_dataset/queries.json', 'r') as f:
    data = json.load(f)

# Convert labels from strings to integers
# DECOMPOSE (Class 1) and DIRECT (Class 0)
for item in data:
    item['label'] = 1 if item['label'] == 'DECOMPOSE' else 0

df = pd.DataFrame(data)
print(f"Loaded {len(df)} queries.")
df.head()

Loaded 200 queries.


,text,label
0,Should we implement speculative decoding or mi...,1
1,Give me the architecture of Yi.,0
2,can Yi beat Falcon at HumanEval?,1
3,is it better to use Hugging Face over Triton f...,1
4,Can you provide the inference latency for Mist...,0


In [10]:
# 2. Initialize your fast, low-latency embedding model
# This matches what you'll use in production to stay under 100ms
embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 3. Generate vectors for the training set
print("Generating embeddings...")
embeddings_generator = embedding_model.embed(df["text"].tolist())
X = np.array(list(embeddings_generator))
y = df["label"].values

print(f"X is now a matrix of shape {X.shape} representing the semantic meaning of the queries")
print(f"y is a 1D array of shape {y.shape}")

Generating embeddings...
X is now a matrix of shape (200, 384) representing the semantic meaning of the queries
y is a 1D array of shape (200,)


### Step 5: Train and Save the 1ms Classifier

With the arrays `X` and `y` ready, we will train a Logistic Regression model using `scikit-learn`.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib

# Split into train/test to check accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the classifier
classifier = LogisticRegression(C=1.0, max_iter=1000)
classifier.fit(X_train, y_train)

# Check performance
accuracy = classifier.score(X_test, y_test)
print(f"Classifier Test Accuracy: {accuracy * 100:.2f}%")

# Save the trained model artifact to disk
joblib.dump(classifier, "../data/router_dataset/query_router_model.joblib")
print("Model saved to ../data/router_dataset/query_router_model.joblib")

Classifier Test Accuracy: 100.00%
Model saved to ../data/router_dataset/query_router_model.joblib


### Step 6: Test the Classifier

Let's test the trained classifier on some unseen queries to verify its predictions and latency.

In [14]:
import time

# Test queries
test_queries = [
    "What is the difference between state space models vs transformers",
    "how does back propogation work?"
]

print("Testing classifier on new queries...")
for q in test_queries:
    t0 = time.perf_counter()
    vec = list(embedding_model.embed([q]))[0]
    pred = classifier.predict([vec])[0]
    t1 = time.perf_counter()
    
    label_str = "DECOMPOSE" if pred == 1 else "DIRECT"
    latency_ms = (t1 - t0) * 1000
    print(f"Query: '{q}'\n -> Predicted: {label_str} (Latency: {latency_ms:.2f}ms)\n")


Testing classifier on new queries...
Query: 'What is the difference between state space models vs transformers'
 -> Predicted: DECOMPOSE (Latency: 5.32ms)

Query: 'how does back propogation work?'
 -> Predicted: DIRECT (Latency: 4.18ms)

